In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

In [0]:
from pyspark.sql.functions import col, upper, row_number, expr
from pyspark.sql.window import Window

consumer_mapping = {
    "customer_id": "id_consumidor",
    "customer_zip_code_prefix": "prefixo_cep",
    #OBS: customer_name não existe no dataset, porém como estava na atividade inclui no dicionário de mapeamento. Quando o rename executar, nada acontecerá com esta coluna
    "customer_name": "nome_consumidor",
    "customer_city": "cidade",
    "customer_state": "estado"
}

df = spark.table("bronze.tb_customers")

for old_col, new_col in consumer_mapping.items():
    df = df.withColumnRenamed(old_col, new_col)

window = Window.partitionBy("id_consumidor").orderBy(col("timestamp_ingestion").desc())

#Coluna customer_unique_id não consta no mapeamento, então foi removida
df = df.withColumn("row_num", row_number().over(window)) \
       .filter(col("row_num") == 1) \
       .drop("row_num", "timestamp_ingestion", "customer_unique_id")

#Os estados já estão em Uppercase no dataset, mas para garantir a padronização faço o upper() de qualquer forma
df = df.withColumn("cidade", upper(col("cidade"))) \
       .withColumn("estado", upper(col("estado")))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_consumidores")

In [0]:
from pyspark.sql.functions import when, datediff, to_timestamp

orders_mapping = {
    "order_id": "id_pedido",
    "customer_id": "id_consumidor",
    "order_status": "status",
    "order_purchase_timestamp": "data_hora_pedido",
    "order_approved_at": "data_aprovacao_pedido",
    "order_delivered_carrier_date": "data_entrega_transportadora",
    "order_delivered_customer_date": "data_entrega_consumidor",
    "order_estimated_delivery_date": "data_entrega_estimada"
}

df = spark.table("bronze.tb_orders")

for old_col, new_col in orders_mapping.items():
    df = df.withColumnRenamed(old_col, new_col)

df = df.drop("timestamp_ingestion")

df = df.withColumn("status",
    when(col("status") == "delivered", "entregue")
    .when(col("status") == "canceled", "cancelado")
    .when(col("status") == "shipped", "enviado")
    .when(col("status") == "processing", "processando")
    .when(col("status") == "invoiced", "faturado")
    .when(col("status") == "unavailable", "indisponível")
    .when(col("status") == "created", "criado")
    .when(col("status") == "approved", "aprovado")
    .otherwise("desconhecido")
)

df = df.withColumn("tempo_entrega_dias", datediff(col("data_entrega_consumidor"), col("data_hora_pedido")))

df = df.withColumn("tempo_entrega_estimado_dias", datediff(col("data_entrega_estimada"), col("data_hora_pedido")))

df = df.withColumn("diferenca_entrega_dias", datediff(col("data_entrega_consumidor"), col("data_entrega_estimada")))

df = df.withColumn("entrega_no_prazo",
    when(col("status") != "entregue", "Não Entregue")
    .when(col("diferenca_entrega_dias") <= 0, "Sim")
    .otherwise("Não")
)

#Em caso de o inferSchema não ter identificado corretamente as colunas de data
colunas_data = [
    "data_hora_pedido",
    "data_aprovacao_pedido",
    "data_entrega_transportadora",
    "data_entrega_consumidor",
    "data_entrega_estimada"
]

for c in colunas_data:
    df = df.withColumn(c, to_timestamp(col(c)))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fat_pedidos")

In [0]:
order_items_mapping = {
  "order_id": "id_pedido",
  "order_item_id": "id_item",
  "product_id": "id_produto",
  "seller_id": "id_vendedor",
  "price": "preco_BRL",
  "freight_value": "preco_frete"
}

df = spark.table("bronze.tb_order_items")

for old_col, new_col in order_items_mapping.items():
  df = df.withColumnRenamed(old_col, new_col)

#Coluna Shipping_limit_date não consta no mapeamento, então foi removida
df = df.drop("shipping_limit_date", "timestamp_ingestion")

number_type_mapping = [
  "preco_BRL",
  "preco_frete"
]

for i in number_type_mapping:
  df = df.withColumn(i, expr(f"try_cast({i} as decimal(10,2))"))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fat_itens_pedidos")

In [0]:
order_payments_mapping = {
  "order_id": "id_pedido",
  "payment_sequential": "sequencial_pagamento",
  "payment_type": "tipo_pagamento",
  "payment_installments": "parcelas_pagamento",
  "payment_value": "valor_pagamento"
}

df = spark.table("bronze.tb_order_payments")

for old_col, new_col in order_payments_mapping.items():
  df = df.withColumnRenamed(old_col, new_col)

df = df.drop("timestamp_ingestion")

df = df.withColumn("tipo_pagamento",
    when(col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
    .when(col("tipo_pagamento") == "boleto", "Boleto")
    .when(col("tipo_pagamento") == "voucher", "Voucher")
    .when(col("tipo_pagamento") == "debit_card", "Cartão de Débito")
    .when(col("tipo_pagamento") == "not_defined", "Não Definido")
    .otherwise("desconhecido")
)

number_type_mapping = [
    "valor_pagamento"
]

for i in number_type_mapping:
  df = df.withColumn(i, expr(f"try_cast({i} as decimal(10,2))"))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fat_pagamentos_pedidos")

In [0]:
from pyspark.sql.functions import try_to_timestamp, current_timestamp

order_reviews_mapping ={
    "review_id": "id_avaliacao",
    "order_id": "id_pedido",
    "review_score": "nota_avaliacao",
    #As proximas duas tabelas estão com o mesmo nome no mapeamento. Usarei os nomes sugeridos no grupo do whatsapp
    "review_comment_title": "titulo_avaliacao_comentario",
    "review_comment_message": "mensagem_avaliacao_comentario",
    "review_creation_date": "data_criacao_avaliacao",
    "review_answer_timestamp": "data_resposta_avaliacao"
}

df = spark.table("bronze.tb_order_reviews")

for old_col, new_col in order_reviews_mapping.items():
  df = df.withColumnRenamed(old_col, new_col)

df = df.drop("timestamp_ingestion")

#Garantir tipagem
df = df.withColumn("nota_avaliacao", expr("try_cast(nota_avaliacao as integer)"))

df = df.filter(col("id_pedido").isNotNull())

df = df.withColumn("data_criacao_avaliacao", try_to_timestamp(col("data_criacao_avaliacao"))) \
       .withColumn("data_resposta_avaliacao", try_to_timestamp(col("data_resposta_avaliacao")))

df = df.filter(col("data_criacao_avaliacao") <= current_timestamp())

#Alternativa: Utilizar a função fillna, porém utilizei o when para manter a consistência
df = df.withColumn("titulo_avaliacao_comentario",
    when(col("titulo_avaliacao_comentario").isNull(), "Sem título")
    .otherwise(col("titulo_avaliacao_comentario"))
)

df = df.withColumn("mensagem_avaliacao_comentario",
    when(col("mensagem_avaliacao_comentario").isNull(), "Sem comentário")
    .otherwise(col("mensagem_avaliacao_comentario"))
)

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fat_avaliacoes_pedidos")

In [0]:
from pyspark.sql.functions import try_to_timestamp, coalesce

products_mapping = {
    "product_id": "id_produto",
    #OBS: product_name não existe no dataset, porém como estava na atividade inclui no dicionário de mapeamento. Quando o rename executar, nada acontecerá com esta coluna
    "product_name": "nome_produto",
    "product_category_name": "categoria_produto",
    "product_name_lenght": "tamanho_nome_produto",
    "product_description_lenght": "tamanho_descricao_produto",
    "product_photos_qty": "quantidade_fotos",
    "product_weight_g": "peso_produto_gramas",
    "product_length_cm": "comprimento_centimetros",
    "product_height_cm": "altura_centimetros",
    "product_width_cm": "largura_centimetros"
}

df = spark.table("bronze.tb_products")

for old_col, new_col in products_mapping.items():
  df = df.withColumnRenamed(old_col, new_col)

window = Window.partitionBy("id_produto").orderBy(col("timestamp_ingestion").desc())

df = df.withColumn("row_num", row_number().over(window)) \
       .filter(col("row_num") == 1) \
       .drop("row_num", "timestamp_ingestion")

df = df.fillna({"categoria_produto": "Sem Categoria"})

int_columns = ["tamanho_nome_produto", "tamanho_descricao_produto", "quantidade_fotos"]
decimal_columns = ["peso_produto_gramas", "comprimento_centimetros", "altura_centimetros", "largura_centimetros"]

for c in int_columns:
    df = df.withColumn(c, expr(f"try_cast({c} as integer)"))

for c in decimal_columns:
    df = df.withColumn(c, expr(f"try_cast({c} as decimal(10,2))"))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_produtos")

In [0]:
sellers_mapping = {
    "seller_id": "id_vendedor",
    "seller_name": "nome_vendedor",
    "seller_zip_code_prefix": "prefixo_cep",
    "seller_city": "cidade",
    "seller_state": "estado"
}

df = spark.table("bronze.tb_sellers")

for old_col, new_col in sellers_mapping.items():
  df = df.withColumnRenamed(old_col, new_col)

window = Window.partitionBy("id_vendedor").orderBy(col("timestamp_ingestion").desc())

df = df.withColumn("row_num", row_number().over(window)) \
       .filter(col("row_num") == 1) \
       .drop("row_num", "timestamp_ingestion")

df = df.withColumn("cidade", upper(col("cidade")))
df = df.withColumn("estado", upper(col("estado")))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_vendedores")

In [0]:
product_category_name_translation_mapping = {
    "product_category_name": "nome_produto_pt",
    "product_category_name_english": "nome_produto_en"
}

df = spark.table("bronze.tb_product_category_name_translation")

for old_col, new_col in product_category_name_translation_mapping.items():
  df = df.withColumnRenamed(old_col, new_col)

df = df.drop("timestamp_ingestion")

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_categoria_produtos_traducao")


In [0]:
from pyspark.sql.functions import to_date, last, explode, sequence, min, max

df_dolar = spark.table("bronze.tb_cotacao_dolar")

df_dolar = df_dolar.withColumn("data", to_date(col("dataHoraCotacao"))) \
                   .withColumnRenamed("cotacaoCompra", "cotacao_compra") \
                   .select("data", "cotacao_compra")

data_min = df_dolar.select(min("data")).collect()[0][0]
data_max = df_dolar.select(max("data")).collect()[0][0]

df_calendar = spark.sql(f"""SELECT explode(sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)) as data""")

df_full = df_calendar.join(df_dolar, on="data", how="left")

window = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, 0)

df_full = df_full.withColumn("cotacao_compra",
    last("cotacao_compra", ignorenulls=True).over(window)
)

df_full = df_full.withColumn("cotacao_compra", col("cotacao_compra").cast("decimal(10,4)"))

df_full.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
from pyspark.sql.functions import sum as _sum, round

df_pedidos = spark.table("silver.fat_pedidos")
df_pagamentos = spark.table("silver.fat_pagamentos_pedidos")
df_cotacao = spark.table("silver.dim_cotacao_dolar")

df_pagamentos_agg = df_pagamentos.groupBy("id_pedido") \
    .agg(_sum("valor_pagamento").alias("valor_total_pago_brl"))

df_total = df_pedidos.join(df_pagamentos_agg, on="id_pedido", how="left")

df_total = df_total.withColumn("data_pedido", to_date(col("data_hora_pedido")))

df_total = df_total.join(df_cotacao, df_total["data_pedido"] == df_cotacao["data"], how="left")

from pyspark.sql.functions import round

df_total = df_total.withColumn("valor_total_pago_usd",
    round(col("valor_total_pago_brl") / col("cotacao_compra"), 2)
).withColumn("valor_total_pago_brl", round(col("valor_total_pago_brl"), 2)) \
.select("id_pedido", "id_consumidor", "status", "valor_total_pago_brl", "valor_total_pago_usd", "data_pedido")

df_total.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fat_pedido_total")


In [0]:
%sql
OPTIMIZE silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido);
OPTIMIZE silver.fat_pedidos ZORDER BY (id_pedido);
OPTIMIZE silver.fat_itens_pedidos ZORDER BY (id_pedido);
OPTIMIZE silver.fat_pagamentos_pedidos ZORDER BY (id_pedido);
OPTIMIZE silver.fat_avaliacoes_pedidos ZORDER BY (id_pedido);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 5323049), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1775335785528, 1775335786074, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null)"
